In [8]:
import pandas as pd

# 1. Load the three separate files
df1 = pd.read_csv("cyberbullying_dataset_10k.csv")
df2 = pd.read_csv("cyberbullying_dataset_10k_chunk.csv")
df3 = pd.read_csv("cyberbullying_dataset_10k_chunk2.csv")

# 2. Combine them (Concatenation)
# This stacks them on top of each other
combined_df = pd.concat([df1, df2, df3], ignore_index=True)

# 3. Shuffle the data
# Crucial: You don't want all 'Bullying' cases from file 1 to be at the top.
# frac=1 means shuffle 100% of the data.
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Save the new master dataset
combined_df.to_csv("2k_Dataset.csv", index=False)

print(f"Successfully merged! New dataset has {len(combined_df)} rows.")

Successfully merged! New dataset has 30000 rows.


In [39]:
df = combined_df
df.to_csv("Cyberbullying_30k_dataset.csv", index=False)
df.sample(20)

,text,language,label
2308,Sukhamaano?,Manglish,0
22404,Let's catch up soon.,English,0
23397,Ninakku onnum varilla.,Manglish,1
25058,ശുഭദിനം!,Malayalam,0
2664,You're the dumbest person I know.,English,1
8511,Let's catch up soon.,English,0
5148,Nobody likes you.,English,1
7790,സുഖമാണോ?,Malayalam,0
11311,ഇന്നലെ കണ്ടില്ലല്ലോ.,Malayalam,0
19043,Janmadinaashamsakal!,Manglish,0


In [4]:
import torch
import re
import pandas as pd
import numpy as np
from torch import nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

# ==========================================
# 1. CONFIGURATION & DATA PREPARATION
# ==========================================
MODEL_CHECKPOINT = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"

# Load your 50k dataset
# Ensure your CSV has 'text' and 'label' columns
df = pd.read_csv("Final_50k_Dataset.csv")

# ==========================================
# 2. IMPROVED PREPROCESSING
# ==========================================
def refined_preprocessing(text):
    """
    Cleans text while preserving emotional cues (punctuation) 
    crucial for bullying detection.
    """
    text = str(text)
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Keep alphanumeric characters and essential punctuation (!, ?, .)
    # This supports Malayalam, English, and Manglish scripts
    text = re.sub(r'[^\w\s!?. ,]', '', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(refined_preprocessing)

# ==========================================
# 3. TOKENIZATION & DATASET
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer.encode_plus(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Stratified split to maintain class balance
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['cleaned_text'].tolist(), 
    df['label'].tolist(), 
    test_size=0.2, 
    random_state=42,
    stratify=df['label'] 
)

train_dataset = CyberbullyingDataset(train_texts, train_labels, tokenizer)
val_dataset = CyberbullyingDataset(val_texts, val_labels, tokenizer)

# ==========================================
# 4. MODEL ARCHITECTURE & LoRA (OPTIMIZED)
# ==========================================
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "Non-Bullying", 1: "Bullying"},
    label2id={"Non-Bullying": 0, "Bullying": 1}
)

# Optimized LoRA for better accuracy on 50k samples
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,            # Rank increased for better capacity
    lora_alpha=32,   # Alpha scaled to 2x Rank
    lora_dropout=0.05,
    target_modules=["query", "value"] # Standard targets for BERT-based models
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters() 

# ==========================================
# 5. CUSTOM WEIGHTED TRAINER
# ==========================================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Giving "Bullying" (Class 1) more weight to fix accuracy issues 
        # caused by dataset imbalance.
        loss_fct = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.5]).to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ==========================================
# 6. TRAINING ARGUMENTS (STABILITY FIXES)
# ==========================================
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=5e-4,           # Optimized for LoRA
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=4,           # Increased epochs for better convergence
    weight_decay=0.01,
    lr_scheduler_type="cosine",    # Smooth learning rate decay
    warmup_ratio=0.1,             # Initial warmup for noisy data
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,  # Keeps the most accurate version
    metric_for_best_model="accuracy",
    logging_dir='./logs',
    fp16=torch.cuda.is_available(), # Faster training on your GPU
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Starting Optimized Training...")
trainer.train()

# ==========================================
# 7. INFERENCE WITH EXPLAINABILITY
# ==========================================
def predict_and_explain(text, model, tokenizer):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    clean_text = refined_preprocessing(text)
    inputs = tokenizer(clean_text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
        logits = outputs.logits
        attentions = outputs.attentions 

    probs = F.softmax(logits, dim=-1)
    prediction_id = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][prediction_id].item()
    prediction_label = model.config.id2label[prediction_id]

    # Attention-Based Importance (Last Layer)
    last_layer_attn = torch.mean(attentions[-1], dim=1)[0, 0, :].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    importance = sorted(
        [(t, s) for t, s in zip(tokens, last_layer_attn) if t not in tokenizer.all_special_tokens],
        key=lambda x: x[1], reverse=True
    )

    print(f"\nText: {text}")
    print(f"Result: {prediction_label} ({confidence:.2%})")
    print("Top Influential Tokens:", [t[0] for t in importance[:3]])

# Quick Test
predict_and_explain("Ninakku vattano, eda bhranthan.", model, tokenizer)

2026-01-29 22:08:44.412 
  command:

    streamlit run C:\Users\pxlaw\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\pxlaw\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pxlaw\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0
2026-01-29 22:09:51.025 Session state does not function when running a script without `streamlit run`


In [ ]:
!streamlit run app.py

In [12]:
import torch
import re
import pandas as pd
from torch import nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

# ==========================================
# 1. LOAD & CLEAN YOUR COMBINED DATASET
# ==========================================
# Ensure your CSV is named exactly this or change the string below
df = pd.read_csv("Cyberbullying_30k_dataset.csv")

# Safety check: Keep only the columns we need for training
# This ignores the 'language' column and any extra index numbers
df = df[['text', 'label']]
df = df.dropna() # Remove any empty rows

# ==========================================
# 2. PREPROCESSING (Social Media Ready)
# ==========================================
def clean_text(text):
    text = str(text)
    # Remove URLs and Mentions
    text = re.sub(r'http\S+|@\w+', '', text)
    # Keep Unicode (Malayalam) and alphanumeric (English/Manglish)
    text = re.sub(r'[^\w\s!?. ,]', '', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

# ==========================================
# 3. TOKENIZATION & PEFT SETUP
# ==========================================
MODEL_CHECKPOINT = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer.encode_plus(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Stratified split: ensures equal % of bullying in Train and Test
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), 
    df['label'].tolist(), 
    test_size=0.2, 
    stratify=df['label'],
    random_state=42
)

train_dataset = CyberbullyingDataset(train_texts, train_labels, tokenizer)
val_dataset = CyberbullyingDataset(val_texts, val_labels, tokenizer)

# ==========================================
# 4. LIGHTWEIGHT ARCHITECTURE (LoRA)
# ==========================================
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, 
    num_labels=2,
    id2label={0: "Non-Bullying", 1: "Bullying"},
    label2id={"Non-Bullying": 0, "Bullying": 1}
)

# 
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["query", "value"] 
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters() 

# ==========================================
# 5. TRAINING WITH WEIGHTED LOSS
# ==========================================
class BullyingTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Penalty weight: 1.0 for Safe, 3.0 for Bullying (makes it highly sensitive)
        loss_fct = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=3e-4, 
    per_device_train_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(), # Uses GPU acceleration
    report_to="none"
)

trainer = BullyingTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("🚀 Starting training on your combined 30k dataset...")
trainer.train()

# ==========================================
# 6. LIVE DEMO FUNCTION (For your Presentation)
# ==========================================
def check_comment(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
        conf = probs[0][pred].item()
    
    label = "🚨 BULLYING" if pred == 1 else "✅ SAFE"
    print(f"\nText: {text}")
    print(f"Result: {label} ({conf:.2%})")

# Sample tests for tomorrow
print("\n--- Running Final Demo Tests ---")
check_comment("Ninakku aashamsakal!")      # Should be Safe
check_comment("Aarkkum ninne ishtalla.")   # Should be Bullying

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [15]:
#Model 1
import torch
import re
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

# ==========================================
# 1. CONFIGURATION & DATA PREPARATION
# ==========================================

# Model Checkpoint: The paper specifies IndicBERT v2 [cite: 12, 107]
MODEL_CHECKPOINT = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"

# Sample Data based on Table II in the PDF [cite: 126, 131]
# In a real scenario, you would load your full CSV here.
# data = [
#     {"text": "സുഖമാണോ?", "label": 0, "lang": "Malayalam"}, # Non-bullying
#     {"text": "മോട്ടോർ വാഹന നിയമം ഭേദഗതി....കൊണ്ട് അങ്ങനെ ഒരു...", "label": 0, "lang": "Malayalam"}, # Non-bullying
#     {"text": "താൻ ഒരു കഴുവേറി ആണ്.", "label": 1, "lang": "Malayalam"}, # Bullying
#     {"text": "ninakku vattano, eda bhranthan.", "label": 1, "lang": "Manglish"}, # Bullying
#     {"text": "verum waste, eda chetta.", "label": 1, "lang": "Manglish"}, # Bullying
#     {"text": "This represents a standard English comment.", "label": 0, "lang": "English"} # Non-bullying
# ]

# df = pd.DataFrame(data)
df = pd.read_csv("Cyberbullying_30k_dataset.csv")

# ==========================================
# 2. PREPROCESSING (Section V.B)
# ==========================================
def minimal_preprocessing(text):
    """
    Paper Section V.B:
    - Basic cleaning (removal of URLs and excessive special characters) [cite: 102]
    - No transliteration of Manglish [cite: 103]
    """
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove excessive special characters (keeping basic punctuation and language scripts)
    # This regex removes characters that are NOT alphanumeric, spaces, or basic punctuation
    # Note: \w matches unicode characters (so Malayalam script is preserved)
    text = re.sub(r'[^\w\s.,!?]', '', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(minimal_preprocessing)

# ==========================================
# 3. TOKENIZATION & DATASET (Section V.B)
# ==========================================

# "Tokenization performed exclusively using the IndicBERT v2 SentencePiece tokenizer" [cite: 104]
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Split Data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['cleaned_text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

train_dataset = CyberbullyingDataset(train_texts, train_labels, tokenizer)
val_dataset = CyberbullyingDataset(val_texts, val_labels, tokenizer)

# ==========================================
# 4. MODEL ARCHITECTURE & LoRA (Section V.C)
# ==========================================

# Load Base Model
# "The system uses a single multilingual backbone, IndicBERT v2" [cite: 107]
id2label = {0: "Non-Bullying", 1: "Bullying"}
label2id = {"Non-Bullying": 0, "Bullying": 1}

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "Non-Bullying", 1: "Bullying"},
    label2id={"Non-Bullying": 0, "Bullying": 1},
    weights_only=False  # <--- THIS IS THE FIX
)

# Apply LoRA (Low-Rank Adaptation)
# "IndicBERT v2 is fine-tuned using Low-Rank Adaptation (LoRA)" 
# "Base model weights remain frozen... only small LoRA low-rank matrices are trained" [cite: 109, 110]
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,            # Rank of the update matrices (Hyperparameter)
    lora_alpha=16,  # LoRA scaling factor
    lora_dropout=0.1,
    # Target modules usually depend on architecture. For ALBERT/BERT-likes: query/value or all linear.
    target_modules=["query", "value", "key", "dense"] 
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters() 
# This confirms "maintaining model compactness" 

# ==========================================
# 5. TRAINING (Section V.E)
# ==========================================

# "Model is trained using cross-entropy loss" [cite: 121] - (Handled automatically by HF Trainer)
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-4, 
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",  # <--- CHANGED FROM evaluation_strategy
    save_strategy="epoch",
    logging_dir='./logs',
    use_cpu=False if torch.cuda.is_available() else True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Start Training
print("Starting Training...")
trainer.train()

# ==========================================
# 6. INFERENCE & EXPLAINABILITY (Section V.D)
# ==========================================

def predict_and_explain(text, model, tokenizer):
    """
    Implements Section V.D:
    1. Class Confidence Score [cite: 117]
    2. Attention-Based Token Importance 
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # Preprocess
    clean_text = minimal_preprocessing(text)
    
    inputs = tokenizer(
        clean_text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True
    ).to(device)

    with torch.no_grad():
        # Request attention weights via output_attentions=True
        outputs = model(**inputs, output_attentions=True)
        logits = outputs.logits
        attentions = outputs.attentions # Tuple of tensors (one per layer)

    # 1. Class Confidence Score
    probs = F.softmax(logits, dim=-1)
    prediction_id = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][prediction_id].item()
    prediction_label = model.config.id2label[prediction_id]

    # 2. Attention-Based Token Importance
    # Get attention from the final layer 
    last_layer_attention = attentions[-1] # Shape: (batch, num_heads, seq_len, seq_len)
    
    # Average attention across all heads
    avg_attention = torch.mean(last_layer_attention, dim=1) # Shape: (batch, seq_len, seq_len)
    
    # Extract attention weights for the [CLS] token (index 0) focusing on other tokens
    # This represents how much the model focused on specific words to make the classification
    cls_attention = avg_attention[0, 0, :].cpu().numpy()
    
    # Map tokens to attention scores
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    # Filter out special tokens ([CLS], [SEP], [PAD]) for clearer explanation
    token_importance = []
    for token, score in zip(tokens, cls_attention):
        if token not in tokenizer.all_special_tokens:
            token_importance.append((token, score))
    
    # Sort by importance
    token_importance.sort(key=lambda x: x[1], reverse=True)

    print(f"Text: {text}")
    print(f"Prediction: {prediction_label}")
    print(f"Confidence: {confidence:.4f}")
    print("Top influential tokens (Explainability):")
    for tok, score in token_importance[:3]: # Show top 3
        print(f"  - {tok}: {score:.4f}")
    print("-" * 30)

# ==========================================
# DEMO
# ==========================================
# Testing with a sample Manglish sentence from Table II [cite: 131]
test_sentence = "ninakku vattano, eda bhranthan." 
predict_and_explain(test_sentence, model, tokenizer)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/IndicBERTv2-MLM-Sam-TLM and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,340,930 || all params: 279,383,812 || trainable%: 0.4800


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Starting Training...


Epoch,Training Loss,Validation Loss
1,0.000000,0.000000
2,0.000000,0.000000
3,0.000000,0.000000


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Text: ninakku vattano, eda bhranthan.
Prediction: Bullying
Confidence: 1.0000
Top influential tokens (Explainability):
  - nina: 0.1523
  - ##kku: 0.0978
  - ed: 0.0759
------------------------------


In [19]:
test_sentence = "nee oru mandan aanu" 
predict_and_explain(test_sentence, model, tokenizer)

Text: nee oru mandan aanu
Prediction: Bullying
Confidence: 1.0000
Top influential tokens (Explainability):
  - ##u: 0.0749
  - ##e: 0.0574
  - ne: 0.0565
------------------------------


In [20]:
# Save model + tokenizer
model.save_pretrained("cyberbully_model")
tokenizer.save_pretrained("cyberbully_model")


('cyberbully_model\\tokenizer_config.json',
 'cyberbully_model\\special_tokens_map.json',
 'cyberbully_model\\tokenizer.json')

In [ ]:
!streamlit run app.py

In [24]:
print(df['label'].unique())
print(df['label'].dtype)


[0 1]
int64


In [40]:
def merge_subwords1(tokens, scores):
    words = []
    current_word = ""
    current_score = 0.0

    for token, score in zip(tokens, scores):
        if token.startswith("##"):
            current_word += token[2:]
            current_score += score
        else:
            if current_word:
                words.append((current_word, current_score))
            current_word = token
            current_score = score

    if current_word:
        words.append((current_word, current_score))

    return words
def predict_and_explain1(text, model, tokenizer):
    """
    Implements Section V.D:
    1. Class Confidence Score [cite: 117]
    2. Attention-Based Token Importance 
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # Preprocess
    clean_text = minimal_preprocessing(text)
    
    inputs = tokenizer(
        clean_text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True
    ).to(device)

    with torch.no_grad():
        # Request attention weights via output_attentions=True
        outputs = model(**inputs, output_attentions=True)
        logits = outputs.logits
        attentions = outputs.attentions # Tuple of tensors (one per layer)

    # 1. Class Confidence Score
    probs = F.softmax(logits, dim=-1)
    prediction_id = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][prediction_id].item()
    prediction_label = model.config.id2label[prediction_id]

    # 2. Attention-Based Token Importance
    # Get attention from the final layer 
    last_layer_attention = attentions[-1] # Shape: (batch, num_heads, seq_len, seq_len)
    
    # Average attention across all heads
    avg_attention = torch.mean(last_layer_attention, dim=1) # Shape: (batch, seq_len, seq_len)
    
    # Extract attention weights for the [CLS] token (index 0) focusing on other tokens
    # This represents how much the model focused on specific words to make the classification
    cls_attention = avg_attention[0, 0, :].cpu().numpy()
    
    # Map tokens to attention scores
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Remove special tokens
    filtered_tokens = []
    filtered_scores = []
    for token, score in zip(tokens, cls_attention):
        if token not in tokenizer.all_special_tokens:
            filtered_tokens.append(token)
            filtered_scores.append(score)

    # 🔥 Merge subwords into full words
    token_importance = merge_subwords1(filtered_tokens, filtered_scores)
    
    # Sort by importance
    token_importance.sort(key=lambda x: x[1], reverse=True)


    print(f"Text: {text}")
    print(f"Prediction: {prediction_label}")
    print(f"Confidence: {confidence:.4f}")
    print("Top influential tokens (Explainability):")
    for tok, score in token_importance[:3]: # Show top 3
        print(f"  - {tok}: {score:.4f}")
    print("-" * 30)

text = "നീ ഒരു തോൽവിയട " 
predict_and_explain1(text, model, tokenizer)

Text: നീ ഒരു തോൽവിയട 
Prediction: Bullying
Confidence: 1.0000
Top influential tokens (Explainability):
  - തൽവയട: 0.1497
  - ഒര: 0.1354
  - ന: 0.0762
------------------------------


In [44]:
pd.set_option('display.max_rows', None)
df.sample(100)

,text,language,label
19927,Aarkkum ninne ishtalla.,Manglish,1
3056,നിനക്ക് സഹായം വേണോ?,Malayalam,0
21491,Innu nalla divasam aakum.,Manglish,0
12715,Janmadinaashamsakal!,Manglish,0
21343,Congrats on your promotion!,English,0
11095,Good luck with your exams!,English,0
22126,Ninakku oru future illa.,Manglish,1
3674,Happy birthday!,English,0
1472,നിനക്ക് ആരും വേണ്ട.,Malayalam,1
5074,Ninte jeevitham oru joke aanu.,Manglish,1


In [45]:
print("hi")

hi


In [1]:
pip install onnx onnxruntime


   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.4 MB 2.6 MB/s eta 0:00:07
   ---- ----------------------------------- 1.8/16.4 MB 3.1 MB/s eta 0:00:05
   ----- ---------------------------------- 2.4/16.4 MB 2.9 MB/s eta 0:00:05
   ------ --------------------------------- 2.6/16.4 MB 2.6 MB/s eta 0:00:06
   ------- -------------------------------- 2.9/16.4 MB 2.7 MB/s eta 0:00:06
   ------- -------------------------------- 3.1/16.4 MB 2.1 MB/s eta 0:00:07
   -------- ------------------------------- 3.4/16.4 MB 2.1 MB/s eta 0:00:07
   ---------- ----------------------------- 4.2/16.4 MB 2.2 MB/s eta 0:00:06
   ---------- ----------------------------- 4.5/16.4 MB 2.1 MB/s eta 0:00:06
   ------------- -------------------------- 5.5/16.4 MB 2.3 MB/s eta 0:00:05
   -------------- -

In [4]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from onnxruntime.quantization import quantize_dynamic, QuantType

# ==========================================
# 1. CONFIGURATION
# ==========================================
MODEL_CHECKPOINT = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
# Pointing directly to the folder you saved!
SAVED_MODEL_PATH = "cyberbully_model"  

ONNX_MODEL_PATH = "cyberbully_model.onnx"
QUANTIZED_ONNX_PATH = "cyberbully_model_int8.onnx"

print("Step 1: Loading Tokenizer and Base Model...")
tokenizer = AutoTokenizer.from_pretrained(SAVED_MODEL_PATH)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "Non-Bullying", 1: "Bullying"},
    label2id={"Non-Bullying": 0, "Bullying": 1},
    weights_only=False
)

# ==========================================
# 2. MERGE LORA WEIGHTS
# ==========================================
print("Step 2: Loading LoRA weights and merging into base model...")
peft_model = PeftModel.from_pretrained(base_model, SAVED_MODEL_PATH)

# Merge the adapter weights into the base model and unload the PEFT wrapper
merged_model = peft_model.merge_and_unload()
merged_model.eval() 
merged_model.to("cpu") # Move to CPU for safe ONNX export

print("Model successfully merged!")

# ==========================================
# ==========================================
# 3. EXPORT TO ONNX
# ==========================================
print(f"Step 3: Exporting model to ONNX format ({ONNX_MODEL_PATH})...")

# Create a dummy input for the ONNX tracer
dummy_text = "This is a dummy sentence for ONNX tracing."
inputs = tokenizer(
    dummy_text, 
    return_tensors="pt", 
    max_length=128, 
    padding="max_length", 
    truncation=True
)

# Export using PyTorch's ONNX exporter (dynamo=False ensures stable legacy export for transformers)
torch.onnx.export(
    merged_model,
    (inputs['input_ids'], inputs['attention_mask']),
    ONNX_MODEL_PATH,
    export_params=True,
    opset_version=14,
    dynamo=False,  # <--- THIS IS THE FIX FOR TRANSFORMERS
    do_constant_folding=True,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'logits': {0: 'batch_size'}
    }
)
print("ONNX export complete!")

# ==========================================
# 4. INT8 QUANTIZATION
# ==========================================
print(f"Step 4: Applying INT8 Dynamic Quantization ({QUANTIZED_ONNX_PATH})...")

quantize_dynamic(
    ONNX_MODEL_PATH,
    QUANTIZED_ONNX_PATH,
    weight_type=QuantType.QInt8
)

print("Quantization complete!")

# ==========================================
# 5. SIZE COMPARISON 
# ==========================================
original_size = os.path.getsize(ONNX_MODEL_PATH) / (1024 * 1024)
quantized_size = os.path.getsize(QUANTIZED_ONNX_PATH) / (1024 * 1024)

print("\n" + "="*50)
print("DEPLOYMENT OPTIMIZATION RESULTS")
print("="*50)
print(f"Original ONNX Model Size : {original_size:.2f} MB")
print(f"Quantized INT8 Model Size: {quantized_size:.2f} MB")
print(f"Size Reduction           : {(1 - (quantized_size/original_size))*100:.2f}%")
print("="*50)

Step 1: Loading Tokenizer and Base Model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/IndicBERTv2-MLM-Sam-TLM and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 2: Loading LoRA weights and merging into base model...
Model successfully merged!
Step 3: Exporting model to ONNX format (cyberbully_model.onnx)...


C:\Users\pxlaw\AppData\Local\Temp\ipykernel_22112\1188811976.py:58: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export complete!
Step 4: Applying INT8 Dynamic Quantization (cyberbully_model_int8.onnx)...


Quantization complete!

DEPLOYMENT OPTIMIZATION RESULTS
Original ONNX Model Size : 1060.94 MB
Quantized INT8 Model Size: 266.01 MB
Size Reduction           : 74.93%
